# Task 1: Conceptual Understanding


## 1. Difference between "Love" and "love" in NLP
In NLP Love and love are treated as different tokens because NLP models are case-sensitive by default.
- Love may appear at the beginning of a sentence or as a proper noun.
- love is generally used as a common verb or noun.
If case normalization (lowercasing) is not applied, the model will assign different representations to both, which can lead to data fragmentation and reduced model performance.

## 2. What happens if stopwords are not removed?

If stopwords are not removed:
- The dataset becomes noisy and less informative
- Increases dimensionality unnecessarily
- Slows down model training
- Reduces model efficiency in tasks like classification
However, stopwords are not always useless — some carry important contextual meaning.

## 3. Two real-world scenarios where removing stopwords can be harmful
- Sentiment Analysis
Example:
"This is not good" → removing "not" becomes "This is good" - Completely changes the meaning
- Search Engines / Question Answering Systems
Example:
"To be or not to be" - Removing stopwords destroys the structure and meaning of the query

## 4. Difference between Stemming and Lemmatization
- Stemming:
A rule-based process that cuts words to their root form
Often produces non-meaningful words
    - Example: "running" → "run", "studies" → "study"
- Lemmatization:
Uses vocabulary and linguistic rules.Lemmatization is more accurate but slower than stemming.
Produces valid, meaningful base words (lemmas)
    - Example: "running" → "run", "better" → "good"


-------------------------------------------------------------------------------------------------

# Task 2:Build Advanced Preprocessing Function

In [4]:
import re

def preprocess_text(text):

    # Handle empty input
    if text == "" or text is None:
        return [], ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove email patterns
    text = re.sub(r'\S+@\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove emojis (basic way)
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # Handle repeated characters (soooo → so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove short words (length ≤ 2), keep "no", "not"
    cleaned_words = []
    for word in words:
        if len(word) > 2 or word == "no" or word == "not":
            cleaned_words.append(word)

    # Join words back to sentence
    cleaned_sentence = " ".join(cleaned_words)

    return cleaned_words, cleaned_sentence




------------------------------------------------------------------


# Task 3: Stress Testing

In [5]:
# Sample inputs
samples = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# Testing loop
for text in samples:
    tokens, sentence = preprocess_text(text)

    print("Original Text   :", text)
    print("Cleaned Tokens  :", tokens)
    print("Cleaned Sentence:", sentence)
    print("-" * 50)

Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
--------------------------------------------------
Original Text   : I absolutely looooved this product 😍😍
Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
--------------------------------------------------
Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
--------------------------------------------------
Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call
--------------------------------------------------
Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
--------------------------------------------------
Original Text   : Visit https://openai.com now!
Cleaned Tokens  : ['visit', 'now']
Cleaned

-----------------------------------------------------------

# Task 4: Token Analytics Code

In [6]:
print("\nTOKEN ANALYTICS\n")

for i, text in enumerate(samples, 1):
    tokens, sentence = preprocess_text(text)

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))

    # Avoid division by zero
    if total_tokens > 0:
        avg_length = sum(len(word) for word in tokens) / total_tokens
    else:
        avg_length = 0

    print(f"Sentence {i}")
    print("Original Text        :", text)
    print("Tokens               :", tokens)
    print("Total Tokens         :", total_tokens)
    print("Unique Tokens        :", unique_tokens)
    print("Average Token Length :", round(avg_length, 2))
    print("-" * 60)


TOKEN ANALYTICS

Sentence 1
Original Text        : Get 100% FREE access now!!!
Tokens               : ['get', 'free', 'access', 'now']
Total Tokens         : 4
Unique Tokens        : 4
Average Token Length : 4.0
------------------------------------------------------------
Sentence 2
Original Text        : I absolutely looooved this product 😍😍
Tokens               : ['absolutely', 'loved', 'this', 'product']
Total Tokens         : 4
Unique Tokens        : 4
Average Token Length : 6.5
------------------------------------------------------------
Sentence 3
Original Text        : Worst service ever... 0/10
Tokens               : ['worst', 'service', 'ever']
Total Tokens         : 3
Unique Tokens        : 3
Average Token Length : 5.33
------------------------------------------------------------
Sentence 4
Original Text        : Call me at 9876543210
Tokens               : ['call']
Total Tokens         : 1
Unique Tokens        : 1
Average Token Length : 4.0
---------------------------------

 ## 1.Which sentence had the most noise?

The sentence with the most noise is:
"Call me at 9876543210" (Sentence 4)

- Reason:
    - Most of the content (phone number) was removed during preprocessing
    - Only one token ("call") remains after cleaning
    - This indicates that the original sentence contained a high amount of non-informative or removable data (numbers)
- This is stronger than picking sentence 9, because your output proves heavy data loss here.

## 2.Which sentence retained the most meaningful tokens after cleaning?

The sentence that retained the most meaningful tokens is:

"I absolutely looooved this product 😍😍" (Sentence 2)

- Reason:
    - After preprocessing → ['absolutely', 'loved', 'this', 'product']
    - All tokens are meaningful and contribute to the sentence’s context
    - It also has the highest average token length (6.5), indicating richer vocabulary

---------------------------------

# Task 5: Frequency Analysis


In [10]:
from collections import Counter

all_tokens = []

# Collect tokens from all sentences
for text in samples:
    tokens, _ = preprocess_text(text)
    all_tokens.extend(tokens)

# Count frequencies
word_freq = Counter(all_tokens)

# Top 10 most frequent
top_10 = word_freq.most_common(10)

# Top 5 least frequent
least_5 = sorted(word_freq.items(), key=lambda x: x[1])[:5]

# Print results
print("Top 10 Most Frequent Words:")
for word, count in top_10:
    print(f"{word} : {count}")

print("\nTop 5 Least Frequent Words:")
for word, count in least_5:
    print(f"{word} : {count}")

Top 10 Most Frequent Words:
this : 4
now : 3
get : 1
free : 1
access : 1
absolutely : 1
loved : 1
product : 1
worst : 1
service : 1

Top 5 Least Frequent Words:
get : 1
free : 1
access : 1
absolutely : 1
loved : 1


--------------------------

# Task 6: Build Full Pipeline



In [11]:
def full_pipeline(text_list):

    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, sentence = preprocess_text(text)

        all_tokens.append(tokens)
        clean_sentences.append(sentence)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

In [12]:
result = full_pipeline(samples)

print("Tokens:")
for t in result["tokens"]:
    print(t)

print("\nClean Sentences:")
for s in result["clean_sentences"]:
    print(s)

Tokens:
['get', 'free', 'access', 'now']
['absolutely', 'loved', 'this', 'product']
['worst', 'service', 'ever']
['call']
['this', 'the', 'best', 'course']
['visit', 'now']
['no', 'this', 'bad']
['got']
['win', 'now', 'limited', 'offer']
['not', 'happy', 'with', 'this']

Clean Sentences:
get free access now
absolutely loved this product
worst service ever
call
this the best course
visit now
no this bad
got
win now limited offer
not happy with this


---------------------------------

# Task 7: Error Handling


In [13]:
test_cases = [
    "",                 # Empty string
    "😂😂😂",            # Only emojis
    "1234567890"        # Only numbers
]

for i, text in enumerate(test_cases, 1):
    tokens, sentence = preprocess_text(text)

    print(f"Test Case {i}")
    print("Original Text   :", repr(text))
    print("Tokens          :", tokens)
    print("Cleaned Sentence:", sentence)
    print("-" * 50)

Test Case 1
Original Text   : ''
Tokens          : []
Cleaned Sentence: 
--------------------------------------------------
Test Case 2
Original Text   : '😂😂😂'
Tokens          : []
Cleaned Sentence: 
--------------------------------------------------
Test Case 3
Original Text   : '1234567890'
Tokens          : []
Cleaned Sentence: 
--------------------------------------------------
